In [1]:
from osgeo import gdal
import pandas as pd
import numpy as np
import os 
import matplotlib.pyplot as plt

In [2]:
ds = gdal.Open(r"C:\Users\easan\Downloads\Canada - 125m - RAD - Equivalent Uranium eU - 2025_Aug\Canada - 125m - RAD - Equivalent Uranium eU - 2025_Aug.TIF")

C:\Users\easan\anaconda3b\Lib\site-packages\osgeo\gdal.py:330: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


In [3]:
xyz = gdal.Translate("can_eq_ura.xyz",ds)
xyz = None

In [4]:
df_eq_ura = pd.read_csv("can_eq_ura.xyz", sep = " ", header = None)
print(df_eq_ura.sample(n=10))


                0          1    2
326822   131187.5  -437437.5  255
60777    906312.5  1901187.5  255
277699  -617437.5    -6812.5  255
251622  1456187.5   225062.5  255
240732  -756562.5   317812.5  255
268019   190812.5    79312.5   65
35799    270312.5  2119812.5  255
264548  2171687.5   112437.5  255
97497   -590937.5  1576562.5   25
289695 -1067937.5  -112812.5  255


In [5]:
df_eq_ura.columns = ["x","y", "value"]
df_eq_ura.to_csv("can_eq_ura.csv",index=False)

In [6]:
print(df_eq_ura.value.unique())


[255   0 100  85 243  45  80 188  19 206   3  60 110 140 194  65  11  25
 225  90   5  15  40  75 105  30 212 150 169  10  50 120  55  35 252 163
 135 157 246 200 249 130 218 175 182  70 115  95  20 125 145 237 231]


In [7]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from pyproj import CRS


def add_fsa_from_easting_northing(
    df,
    easting_col,
    northing_col,
    input_crs_epsg,
    fsa_shapefile_path,
    fsa_code_column="CFSAUID"
):
    """
    Adds a column 'FSA' to df based on easting/northing coordinates.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe
    easting_col : str
        Name of easting column
    northing_col : str
        Name of northing column
    input_crs_epsg : int
        EPSG code of input coordinates (e.g., 26917 for NAD83 / UTM zone 17N)
    fsa_shapefile_path : str
        Path to FSA boundary shapefile
    fsa_code_column : str
        Column name in shapefile containing FSA codes

    Returns
    -------
    pandas.DataFrame
        Original dataframe with new column 'FSA'
    """

    # Copy dataframe
    df_copy = df.copy()

    # Create geometry from easting/northing
    geometry = [
        Point(xy) for xy in zip(df_copy[easting_col], df_copy[northing_col])
    ]

    gdf_points = gpd.GeoDataFrame(
        df_copy,
        geometry=geometry,
        crs=CRS.from_epsg(input_crs_epsg)
    )

    # Load FSA boundaries
    gdf_fsa = gpd.read_file(fsa_shapefile_path)

    # Reproject points to match FSA CRS if necessary
    if gdf_points.crs != gdf_fsa.crs:
        gdf_points = gdf_points.to_crs(gdf_fsa.crs)

    # Spatial join
    joined = gpd.sjoin(
        gdf_points,
        gdf_fsa[[fsa_code_column, "geometry"]],
        how="left",
        predicate="within"
    )

    # Add FSA column
    df_copy["FSA"] = joined[fsa_code_column].values

    return df_copy

In [8]:
new_df = add_fsa_from_easting_northing(df_eq_ura,'x','y',3978,r"C:\Users\easan\Downloads\lfsa000b21a_e\lfsa000b21a_e\lfsa000b21a_e.shp")

In [9]:
print(new_df.head())


           x          y  value  FSA
0 -2121312.5  2431187.5    255  NaN
1 -2114687.5  2431187.5    255  NaN
2 -2108062.5  2431187.5    255  NaN
3 -2101437.5  2431187.5    255  NaN
4 -2094812.5  2431187.5    255  NaN


In [10]:
print(new_df.FSA.unique())

[nan 'Y0B' 'X0E' ... 'N9V' 'N8H' 'N9Y']


In [11]:
new_df.to_csv("can_ura_updated.csv",index=False)

In [12]:
radon_df = pd.read_csv(r"C:\Users\easan\Downloads\radon-concentration.csv")

In [13]:
radon_df = radon_df.rename(columns={'ForwardSortationAreaCodes':'FSA'})

In [14]:
rad_ura = pd.merge(radon_df,new_df, on='FSA', how='left')

In [48]:
print(radon_df.columns)

Index(['ResultNumber', 'ProvinceTerritory', 'Health Region2007',
       'HealthRegionCode2007', 'ForwardSortationAreaCodes',
       'TestDurationInDays', 'AverageRadonConcentrationInBqPerM3',
       'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11',
       'Unnamed: 12'],
      dtype='object')


In [15]:
print(rad_ura.head())

   ResultNumber ProvinceTerritory  \
0           1.0                NL   
1           2.0                NL   
2           3.0                NL   
3           3.0                NL   
4           3.0                NL   

                              Health Region2007  HealthRegionCode2007  FSA  \
0  Eastern Regional Integrated Health Authority                1011.0  A0A   
1  Eastern Regional Integrated Health Authority                1011.0  A0A   
2  Eastern Regional Integrated Health Authority                1011.0  A0E   
3  Eastern Regional Integrated Health Authority                1011.0  A0E   
4  Eastern Regional Integrated Health Authority                1011.0  A0E   

   TestDurationInDays AverageRadonConcentrationInBqPerM3  Unnamed: 7  \
0               127.0                                 20         NaN   
1               108.0                                 36         NaN   
2                91.0                                <15         NaN   
3                91.

In [16]:
print(len(rad_ura))

10630114


In [17]:
rad_ura.to_csv("rad_ura.csv",index=False)